## Test the greedy layout design strategy

In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *

result_dir = f"result/layout/greedy/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

package = Package((50, 50), FabricateModel(20, 10), design_strategy="greedy", result_dir=result_dir)
# package = Package(design_strategy="greedy", result_dir=result_dir)

package.save_design_history()
package.save()
print("Package score:", package.estimate_score())

Initial score:  33.878021187989866
New action:  (ComputeModule_1_1_48_48, Strip, Y, 4)
New score:  80.83825999419155
New action:  (ComputeModule_31_1_34_48, Strip, X, 20)
New score:  104.43789658919748
Package score:  86.76993159278022


In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.element.package import *

result_dir = f"result/layout/greedy/example_20250617111701"
package = Package(pre_designed_package_path=os.path.join(result_dir, "package.pkl"))
print("Package score:", package.estimate_score())

## Test the rl-based layout design strategy

In [ ]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.layout.env import QuNoCEnv
from qunoc.layout.agent import LayoutAgent

result_dir = f"result/layout/rl/example_{get_timestamp()}"
os.makedirs(result_dir, exist_ok=True)

env = QuNoCEnv((50, 50), FabricateModel(20, 10), result_dir=result_dir)
# env = QuNoCEnv(result_dir=result_dir)
agent = LayoutAgent(env, result_dir=result_dir)
agent.train(1000)
package = agent.design()

package.save_design_history()
package.save()
print("Package score:", package.estimate_score())

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Training agent...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 16.9     |
|    ep_rew_mean     | -1.4e+09 |
| time/              |          |
|    fps             | 8        |
|    iterations      | 1        |
|    time_elapsed    | 249      |
|    total_timesteps | 2048     |
---------------------------------
Designing layout...
Initial score:  39.61728499372216
New action:  [17 40  2  1  1]
New score:  74.54653705609819
New action:  [28 23  1  0  3]
New score:  65.51810600817757
New action:  [28 23  1  0  3]
New score:  71.03631243430449
New action:  [28 23  1  0  3]
New score:  70.1210073894442
New action:  [28 23  1  0  3]
New score:  80.50475805612783
New action:  [28 23  1  0  3]
New score:  70.70797684069667
New action:  [28 23  1  0  3]
New score:  68.75900348327541
New action:  [28 23  1  0  3]
New score:  67.74360363335184
New action:  [2

## Test the benchmark generation

In [1]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.WARN)

from qunoc.element.benchmark import *

vqe_program = vqe(5)
print(vqe_program)
qft_program = qft(5)
print(qft_program)
qaoa_program = qaoa(5)
print(qaoa_program)
bv_program = bv(5)
print(bv_program)

vqe program: Control_0_1, Control_0_2, Control_0_3, Control_0_4, Control_1_2, Control_1_3, Control_1_4, Control_2_3, Control_2_4, Control_3_4
qft program: Control_1_0, Control_2_0, Control_3_0, Control_2_1, Control_4_0, Control_3_1, Control_4_1, Control_3_2, Control_4_2, Control_4_3
qaoa program: Control_2_4, Control_3_4, Control_0_2, Control_1_4, Control_1_2
bv program: Control_0_3, Control_0_4


## Test the routing algorithm

In [1]:
import os
import sys
sys.path.append('..')
import logging
logging.basicConfig(level=logging.INFO)

from qunoc.utils.utils import get_timestamp
from qunoc.element.package import *
from qunoc.element.benchmark import *
from qunoc.routing.router import *

# result_dir = f"result/routing/example_{get_timestamp()}"
result_dir = f"result/routing/example"
os.makedirs(result_dir, exist_ok=True)

package = Package(package_size=(9, 9), fab_model=FabricateModel(9, 9), design_strategy="manual", result_dir=result_dir)
package.design_module_top_down(package.compute_modules[0], DesignTemplate.Grid, DesignOrientation.X, 3)

package.save_design_history()
package.save()
print("Package score:", package.estimate_score(), "\n")

program = vqe(10)
gate_depend_fig = program.draw_depend()
gate_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_gate_depend.jpg"), bbox_inches='tight')
qiskit_circuit_fig = program.draw_circuit()
qiskit_circuit_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_qiskit_circuit.jpg"), bbox_inches='tight')

router = Router(package=package, program=program, result_dir=result_dir)
router.map_qubits()
router.compile_blocks_and_actions()
router.save_block_history()
block_depend_fig = router.draw_block()
block_depend_fig.savefig(os.path.join(result_dir, f"{program.algo_name}_{program.data_qubit_num}_block_depend.jpg"), bbox_inches='tight')

# router.schedule_actions_to_circuit()

# circuit = router.route_program_on_package()
# print(f"{program.algo_name} circuit: \n{circuit}")
# circuit.save(f"{result_dir}{program.algo_name}_circuit.pkl")
# print(f"{program.algo_name} routing result: QST_Fidelity_{router.estimate_average_qst_fide()}, Depth_{router.estimate_depth()}")

Package score: 177.38246992177378 



INFO:root:Initial qubit mapping: {0: (3, 7), 1: (2, 2), 2: (1, 6), 3: (2, 5), 4: (6, 2), 5: (7, 7), 6: (6, 5), 7: (3, 3), 8: (5, 6), 9: (3, 6)}
INFO:root:
INFO:root:Constructed TeleBlock 0: Teleport qubit 0 to [1, 2, 3, 4, 5, 6, 7, 8, 9].
INFO:root:Transformed GhzPreparation 0: Prepare 10-GHZ state for logic qubits [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] on physical qubits [(0, 1), (0, 7), (2, 4), (0, 4), (3, 4), (0, 3), (1, 4), (0, 6), (0, 2), (0, 5)].
INFO:root:Swap path for 0: [(3, 7), (2, 7), (1, 7)]
INFO:root:Transformed SwapGate 1: Swap gate between physical qubits (3, 7) and (2, 7).
INFO:root:Transformed Transport 2: Transport logic qubit 0 from physical qubit (0, 1) to physical qubit (0, 7).
INFO:root:Transformed Interchange 3: Interchange logic qubit 0 between physical qubits ((0, 7), (0, 8)).
INFO:root:Transformed Transport 4: Transport logic qubit 0 from physical qubit (0, 8) to physical qubit (1, 8).
INFO:root:Transformed Alight 5: Alight logic qubit 0 from physical qubit (1, 8) to 

AssertionError: phy_qubit_v: (7, 5), local_gate.target_qubit: 9